# Prompt 12: End-to-End JumpGuard Pipeline

This notebook demonstrates the Prompt 12 orchestration layer. It coordinates existing video processing, pose estimation, landmark-derived feature extraction, run-local reporting, dashboard packaging, and metadata export. It does not add new algorithms, feature calculations, machine-learning predictions, or clinical interpretations.


In [4]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import JumpGuardPipeline


In [5]:
video_path = PROJECT_ROOT / "data" / "sample" / "jump.mp4"
output_root = PROJECT_ROOT / 'reports'
assert video_path.exists(), video_path
video_path


PosixPath('/Users/arnavanney/Documents/JumpguardAI/data/sample/jump.mp4')

In [6]:
pipeline = JumpGuardPipeline()
result = pipeline.process_video(
    video_path,
    output_root,
    run_id='run_demo_pipeline',
    frame_skip=6,
)
result.run_directory


I0000 00:00:1783657952.177163 41674951 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
I0000 00:00:1783657952.306565 41674951 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1783657952.388280 41674955 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783657952.401469 41674955 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783657952.461241 41674957 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


PosixPath('/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline')

In [7]:
metadata = json.loads(Path(result.generated_files['metadata']).read_text(encoding='utf-8'))
{
    'status': metadata['status'],
    'frame_count': metadata['frame_count'],
    'feature_count': metadata['feature_count'],
    'predictions_generated': metadata['predictions_generated'],
    'run_directory': str(result.run_directory),
}


{'status': 'success',
 'frame_count': 92,
 'feature_count': 57,
 'predictions_generated': False,
 'run_directory': '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline'}

In [8]:
feature_table = pd.read_csv(result.run_directory / 'features' / 'feature_table.csv')
feature_table.head()


,participant_id,trial_slot,trial_name,condition,is_empty,hip_flexion_right_mean,hip_flexion_right_median,hip_flexion_right_std,hip_flexion_right_variance,hip_flexion_right_maximum,...,ankle_angle_left_time_to_peak,hip_flexion_rom_absolute_difference,hip_flexion_rom_percent_difference,hip_flexion_rom_symmetry_index,knee_flexion_rom_absolute_difference,knee_flexion_rom_percent_difference,knee_flexion_rom_symmetry_index,ankle_angle_rom_absolute_difference,ankle_angle_rom_percent_difference,ankle_angle_rom_symmetry_index
0,NaN,1,jump.mp4,uploaded,False,60.833066,43.34835,31.749211,1008.012416,109.481204,...,1.6,12.782886,13.928711,13.928711,1.679717,2.287478,-2.287478,2.934753,6.766478,-6.766478


In [9]:
sorted(result.generated_files.items())


[('athlete_report_html',
  '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline/athlete_report/athlete_report.html'),
 ('athlete_report_json',
  '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline/athlete_report/athlete_report.json'),
 ('athlete_report_markdown',
  '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline/athlete_report/athlete_report.md'),
 ('dashboard_html',
  '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline/dashboard/index.html'),
 ('dashboard_json',
  '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline/dashboard/dashboard_payload.json'),
 ('features_feature_statistics',
  '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline/features/feature_statistics.json'),
 ('features_feature_table_csv',
  '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pipeline/features/feature_table.csv'),
 ('features_feature_table_json',
  '/Users/arnavanney/Documents/JumpguardAI/reports/run_demo_pi

The dashboard is written to `reports/run_demo_pipeline/dashboard/index.html`, and the run-local descriptive report is written to `reports/run_demo_pipeline/athlete_report/`.
